# Slide Generation with Gemini API

This notebook generates a PowerPoint presentation from a JSON input using the Gemini API.

In [17]:
!pip install python-pptx google-generativeai

## Import Libraries

In [18]:
import google.generativeai as genai
from pptx import Presentation
import json
import os

## Configure Gemini API

In [ ]:
gemini_api_key = "blah"  # Replace with your Gemini API key
genai.configure(api_key=gemini_api_key)

## Load JSON Input

In [20]:
json_input = """ 
{
  "title": "The Future of AI",
  "slides": [
    {
      "title": "Introduction",
      "points": [
        "What is AI?",
        "Brief history of AI",
        "Importance of AI in the modern world"
      ]
    },
    {
      "title": "AI in Healthcare",
      "points": [
        "Drug discovery and development",
        "Personalized medicine",
        "Medical imaging analysis"
      ]
    },
    {
      "title": "AI in Finance",
      "points": [
        "Algorithmic trading",
        "Fraud detection",
        "Credit scoring"
      ]
    },
    {
      "title": "Conclusion",
      "points": [
        "The future is here",
        "Ethical considerations",
        "Q&A"
      ]
    }
  ]
}
"""

data = json.loads(json_input)

## Generate Slide Content with Gemini

In [21]:
import json

model = genai.GenerativeModel('gemini-2.5-pro')

def generate_slide_content(slide_data):
    prompt = f"""
    You are an expert presentation creator. Your task is to generate the content for a single presentation slide.
    The slide title is: "{slide_data['title']}"
    The key points to cover are: {', '.join(slide_data['points'])}

    Based on this, please generate the following in a JSON format:
    1.  A compelling and concise "title" for the slide.
    2.  A "narrative" (2-3 sentences) that introduces the topic in an engaging way.
    3.  A list of "bullet_points" (3-5 points) that expand on the key points provided, making them more detailed and informative. Do not just copy the input points.
    4.  A "visual_suggestion" (e.g., "an icon of a brain with circuits", "a graph showing upward trend").

    Return ONLY the JSON object. Example format:
    {{
      "title": "The Dawn of Artificial Intelligence",
      "narrative": "We begin our journey by exploring the fundamental concepts of AI, from its historical roots to its profound impact on our modern world.",
      "bullet_points": [
        "Defining Intelligence: Differentiating between narrow AI (ANI), general AI (AGI), and superintelligence (ASI).",
        "A Historical Perspective: Key milestones from the Turing Test to the rise of deep learning.",
        "The AI-Powered World: How AI is already integrated into daily life, from recommendation engines to smart assistants."
      ],
      "visual_suggestion": "A timeline graphic with key AI milestones."
    }}
    """
    response = model.generate_content(prompt)
    # Clean up the response to make sure it's valid JSON
    clean_response = response.text.strip().replace('```json', '').replace('```', '')
    try:
        return json.loads(clean_response)
    except json.JSONDecodeError:
        # Fallback in case the response is not valid JSON
        return {
            "title": slide_data['title'],
            "narrative": "Could not generate content.",
            "bullet_points": slide_data['points'],
            "visual_suggestion": "Error icon"
        }

slide_contents = []
for slide in data['slides']:
    print(f"Generating content for slide: {slide['title']}...")
    slide_contents.append(generate_slide_content(slide))

print("All slide content generated.")


Generating content for slide: Introduction...
Generating content for slide: AI in Healthcare...
Generating content for slide: AI in Finance...
Generating content for slide: Conclusion...
All slide content generated.


## Create Presentation

In [22]:
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN

# --- Custom Presentation Design ---

prs = Presentation()
prs.slide_width = Inches(16)
prs.slide_height = Inches(9)

# Define a modern, dark color palette
background_color = RGBColor(18, 18, 18)  # Dark gray
text_color = RGBColor(255, 255, 255)     # White
accent_color = RGBColor(13, 226, 141)    # A vibrant green

# --- Helper function to create a blank slide with a dark background ---
def add_dark_slide(layout_num):
    slide_layout = prs.slide_layouts[layout_num]
    slide = prs.slides.add_slide(slide_layout)
    
    # Set background color
    background = slide.background
    fill = background.fill
    fill.solid()
    fill.fore_color.rgb = background_color
    
    return slide

# --- Title Slide ---
title_slide = add_dark_slide(5) # Using a blank layout

# Add a title
title_shape = title_slide.shapes.add_textbox(Inches(1), Inches(3), Inches(14), Inches(2))
title_text = title_shape.text_frame
title_text.text = data['title']
p = title_text.paragraphs[0]
p.font.name = 'Helvetica Neue'
p.font.size = Pt(60)
p.font.bold = True
p.font.color.rgb = text_color
p.alignment = PP_ALIGN.CENTER

# Add a subtitle
subtitle_shape = title_slide.shapes.add_textbox(Inches(1), Inches(5), Inches(14), Inches(1))
subtitle_text = subtitle_shape.text_frame
subtitle_text.text = "Generated by Gemini 2.5 Pro"
p = subtitle_text.paragraphs[0]
p.font.name = 'Helvetica Neue'
p.font.size = Pt(24)
p.font.color.rgb = accent_color
p.alignment = PP_ALIGN.CENTER

# --- Content Slides ---
for content in slide_contents:
    content_slide = add_dark_slide(5) # Using a blank layout

    # Add slide title
    slide_title_shape = content_slide.shapes.add_textbox(Inches(1), Inches(0.5), Inches(14), Inches(1.5))
    slide_title_text = slide_title_shape.text_frame
    slide_title_text.text = content['title']
    p = slide_title_text.paragraphs[0]
    p.font.name = 'Helvetica Neue'
    p.font.size = Pt(44)
    p.font.bold = True
    p.font.color.rgb = text_color

    # Add narrative
    narrative_shape = content_slide.shapes.add_textbox(Inches(1), Inches(2), Inches(14), Inches(1))
    narrative_text = narrative_shape.text_frame
    narrative_text.text = content.get('narrative', '')
    p = narrative_text.paragraphs[0]
    p.font.name = 'Helvetica Neue'
    p.font.size = Pt(20)
    p.font.color.rgb = text_color

    # Add bullet points
    bullet_points_shape = content_slide.shapes.add_textbox(Inches(1.5), Inches(3.5), Inches(13), Inches(4))
    bullet_points_text = bullet_points_shape.text_frame
    for point in content['bullet_points']:
        p = bullet_points_text.add_paragraph()
        p.text = point
        p.font.name = 'Helvetica Neue'
        p.font.size = Pt(18)
        p.font.color.rgb = text_color
        p.level = 1

    # Add visual suggestion
    visual_suggestion_shape = content_slide.shapes.add_textbox(Inches(1), Inches(8), Inches(14), Inches(0.5))
    visual_suggestion_text = visual_suggestion_shape.text_frame
    visual_suggestion_text.text = f"Visual Suggestion: {content['visual_suggestion']}"
    p = visual_suggestion_text.paragraphs[0]
    p.font.name = 'Helvetica Neue'
    p.font.size = Pt(12)
    p.font.italic = True
    p.font.color.rgb = accent_color

# --- Save the Presentation ---
output_filename = "AI_Generated_Presentation_v2.pptx"
prs.save(output_filename)
print(f"Presentation '{output_filename}' created successfully with a new design.")


Presentation 'AI_Generated_Presentation_v2.pptx' created successfully with a new design.
